# Bangladesh RMG Export Forecasting

A reproducible walkthrough comparing seasonal-naive, Holt-Winters and SARIMAX forecasts using synthetic monthly export series. The series is not official Bangladesh export data.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from rmg_export_forecasting.models import (
    seasonal_naive, fit_holt_winters, fit_sarimax, score_forecast,
    expanding_window_backtest, scenario_adjustment
)

df=pd.read_csv(ROOT/"data/demo/monthly_exports_synthetic.csv",parse_dates=["month"])
series=df.set_index("month")["total_exports_usd_million"].asfreq("MS")
display(df.head())

## 1. Visualise the series and reserve a chronological holdout

In [ ]:
horizon=12
train,test=series.iloc[:-horizon],series.iloc[-horizon:]
plt.figure(figsize=(11,4))
plt.plot(train,label="Training")
plt.plot(test,label="Holdout")
plt.title("Synthetic monthly RMG exports")
plt.ylabel("USD million (synthetic)")
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 2. Fit candidate forecasts

In [ ]:
forecasts={
    "Seasonal naive":seasonal_naive(train,horizon),
    "Holt-Winters":fit_holt_winters(train,horizon),
    "SARIMAX":fit_sarimax(train,horizon)
}
rows=[]
for name,pred in forecasts.items():
    pred.index=test.index
    rows.append({"model":name,**score_forecast(test,pred)})
comparison=pd.DataFrame(rows).sort_values("RMSE")
display(comparison.round(3))

## 3. Expanding-window backtest and scenarios

In [ ]:
backtest=expanding_window_backtest(series,min_train=36,horizon=12)
display(backtest)

best=forecasts["SARIMAX"].copy()
scenario_down= scenario_adjustment(best,-8)
scenario_up=scenario_adjustment(best,5)
plt.figure(figsize=(10,4))
plt.plot(best.index,best,label="SARIMAX")
plt.plot(scenario_down.index,scenario_down,label="Downside -8%")
plt.plot(scenario_up.index,scenario_up,label="Upside +5%")
plt.legend()
plt.title("Illustrative scenario shocks (not causal estimates)")
plt.grid(alpha=.2)
plt.show()

## Interpretation

Compare forecast error using the same holdout and inspect residuals for remaining structure. Scenario shocks are deterministic stress tests, not economic causal effects. Replace the synthetic series with documented EPB/BGMEA records, preserve units and definitions, and validate with rolling-origin backtests.